To demonstrate ``AAPred().eval()``, we obtain the ``DOM_GSEC`` dataset and its feature matrix:

In [1]:
import aaanalysis as aa
aa.options["verbose"] = False  # Disable verbosity

# DOM_GSEC example dataset + its feature set (see [Breimann25]_)
df_seq = aa.load_dataset(name="DOM_GSEC")
labels = df_seq["label"].to_list()
df_feat = aa.load_features(name="DOM_GSEC").head(20)

# Build the CPP feature matrix
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)
X = sf.feature_matrix(features=df_feat["feature"], df_parts=df_parts)

/Users/stephanbreimann/Programming/1Packages/wt-aapred-eval-cv/aaanalysis/feature_engineering/_backend/cpp_run.py:164: UserWarning: CPP is using the Python kernel fallback — the compiled Cython extension is not available in this install. Output is bit-exact with the Cython path but ~2x slower. Reinstall via `pip install --force-reinstall aaanalysis` to fetch a prebuilt wheel.
  warnings.warn(


``eval`` scores every model across metrics by stratified cross-validation and returns a long-format table (one row per model, metric, and evaluation principle):

In [2]:
aap = aa.AAPred(random_state=42)
df_eval = aap.eval(X, labels)
aa.display_df(df_eval, n_rows=10, show_shape=True)

DataFrame shape: (4, 5)


,model,metric,principle,score,score_std
1,RandomForestClassifier,accuracy,cv,0.809231,0.064659
2,RandomForestClassifier,balanced_accuracy,cv,0.810256,0.065416
3,RandomForestClassifier,f1,cv,0.810256,0.059252
4,RandomForestClassifier,roc_auc,cv,0.887278,0.071059


Providing a held-out set adds the ``holdout`` principle beside cross-validation (``score_std`` is ``NaN`` for the single held-out estimate):

In [3]:
from sklearn.model_selection import train_test_split
X_tr, X_ho, y_tr, y_ho = train_test_split(X, labels, test_size=0.3, random_state=0, stratify=labels)
df_eval = aap.eval(X_tr, y_tr, X_holdout=X_ho, labels_holdout=y_ho)
aa.display_df(df_eval, n_rows=10, show_shape=True)

DataFrame shape: (8, 5)


,model,metric,principle,score,score_std
1,RandomForestClassifier,accuracy,cv,0.806536,0.105052
2,RandomForestClassifier,accuracy,holdout,0.842105,nan
3,RandomForestClassifier,balanced_accuracy,cv,0.808333,0.105556
4,RandomForestClassifier,balanced_accuracy,holdout,0.842105,nan
5,RandomForestClassifier,f1,cv,0.800611,0.117547
6,RandomForestClassifier,f1,holdout,0.833333,nan
7,RandomForestClassifier,roc_auc,cv,0.916667,0.078567
8,RandomForestClassifier,roc_auc,holdout,0.896122,nan


The metrics and number of cross-validation folds are controlled by ``metrics`` and ``n_cv``:

In [4]:
df_eval = aap.eval(X, labels, metrics=["balanced_accuracy", "roc_auc"], n_cv=3)
aa.display_df(df_eval, n_rows=10, show_shape=True)

DataFrame shape: (2, 5)


,model,metric,principle,score,score_std
1,RandomForestClassifier,balanced_accuracy,cv,0.793651,0.059391
2,RandomForestClassifier,roc_auc,cv,0.863190,0.061409


Pass ``cv`` to cross-validate with an arbitrary scikit-learn splitter, such as ``LeaveOneOut`` for a small, imbalanced set where the integer ``n_cv`` folds would be capped at the smallest class count. The splitter is not capped, and each metric is scored **once** on the pooled out-of-fold predictions (the ``cv_pooled`` principle, so ``score_std`` is ``NaN``), reproducing ``balanced_accuracy_score(labels, cross_val_predict(estimator, X, labels, cv=cv))``:

In [5]:
from sklearn.model_selection import LeaveOneOut
df_eval = aap.eval(X, labels, cv=LeaveOneOut(), metrics=["balanced_accuracy"])
aa.display_df(df_eval, n_rows=10, show_shape=True)

DataFrame shape: (1, 5)


,model,metric,principle,score,score_std
1,RandomForestClassifier,balanced_accuracy,cv_pooled,0.825397,nan


Set ``baseline`` to compare the CPP features against simple, non-positional **composition baselines** built internally from ``df_seq`` — ``scale`` (scale-average), ``aac`` (amino-acid composition), and ``dpc`` (dipeptide composition), each averaged over ``list_parts``. Every baseline is cross-validated with the same models and folds, and the rows are tagged in a leading ``features`` column (``cpp`` for the bound features), so the whole "CPP vs baseline" comparison comes from one call:

In [6]:
df_eval = aap.eval(X, labels, df_seq=df_seq,
                      baseline=["scale", "aac", "dpc"], list_parts="tmd_jmd")
aa.display_df(df_eval, n_rows=10, show_shape=True)

DataFrame shape: (16, 6)


,features,model,metric,principle,score,score_std
1,cpp,RandomForestClassifier,accuracy,cv,0.809231,0.064659
2,cpp,RandomForestClassifier,balanced_accuracy,cv,0.810256,0.065416
3,cpp,RandomForestClassifier,f1,cv,0.810256,0.059252
4,cpp,RandomForestClassifier,roc_auc,cv,0.887278,0.071059
5,scale,RandomForestClassifier,accuracy,cv,0.754462,0.094264
6,scale,RandomForestClassifier,balanced_accuracy,cv,0.757051,0.095218
7,scale,RandomForestClassifier,f1,cv,0.749096,0.093903
8,scale,RandomForestClassifier,roc_auc,cv,0.845217,0.075511
9,aac,RandomForestClassifier,accuracy,cv,0.706154,0.090286
10,aac,RandomForestClassifier,balanced_accuracy,cv,0.706410,0.091143
